# SinViT-D Fine-tuning
ViT backbone (embed_dim=96, depth=12, num_heads=3) | 2-Phase Transfer Learning

## Hyperparameters

In [ ]:
# ============================================================
# Hyperparameters
# ============================================================
IMG_H, IMG_W   = 240, 320
BATCH_SIZE     = 32

PHASE1_EPOCHS  = 80
PHASE2_EPOCHS  = 120
TOTAL_EPOCHS   = PHASE1_EPOCHS + PHASE2_EPOCHS
LR             = 1e-4
BACKBONE_SCALE = 0.1
WEIGHT_DECAY   = 0.1
DROPOUT        = 0.1
# ============================================================

PRETRAINED_PATH = 'path/to/sinvit_d_places365_320x240_best.pth'  # update this path
SAVE_DIR        = 'path/to/save'               # update this path
CKPT_NAME       = 'best.pth'


## Model

In [ ]:
class ViTNanoDEncoder(nn.Module):
    """
    SinViT-D (embed_dim=96, depth=12, num_heads=3)
    Compatible with CLS pretrained weights (Places365)
    """
    def __init__(self, pretrained_path):
        super().__init__()
        self.vit = VisionTransformer(
            img_size=(IMG_H, IMG_W), patch_size=16,
            embed_dim=96, depth=12, num_heads=3,
            mlp_ratio=4., qkv_bias=True, num_classes=0)
        if pretrained_path and os.path.isfile(pretrained_path):
            ckpt  = torch.load(pretrained_path, map_location='cpu', weights_only=False)
            state = ckpt.get('model', ckpt)
            vit_s = {}
            for k, v in state.items():
                if   k.startswith('encoder.'): vit_s[k[8:]] = v
                elif k.startswith('vit.'):     vit_s[k[4:]] = v
                else:                          vit_s[k]     = v
            miss, unex = self.vit.load_state_dict(vit_s, strict=False)
        else:
        self.vit.pos_embed = nn.Parameter(self.vit.pos_embed.data.clone())

    @property
    def learnable_token(self): return self.vit.cls_token
    @property
    def pos_embed(self):       return self.vit.pos_embed
    @property
    def patch_embed(self):     return self.vit.patch_embed
    @property
    def blocks(self):          return self.vit.blocks
    @property
    def norm(self):            return self.vit.norm

    def forward(self, x): return self.vit(x)  # [B, 96]

class SingleViTNanoDCSI(nn.Module):
    """cam{CAM_NUM} -> SinViT-D [B,96] -> head -> RSSI"""
    def __init__(self, pretrained_path, dropout=DROPOUT):
        super().__init__()
        self.vit  = ViTNanoDEncoder(pretrained_path)
        self.head = nn.Sequential(
            nn.LayerNorm(96), nn.Dropout(dropout),
            nn.Linear(96, 128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, OUT_DIM))

    def forward(self, img):
        return self.head(self.vit(img))  # [B, 1]

    def param_groups(self, lr, backbone_scale):
        bb = {f'vit.vit.{sub}' for sub in ['patch_embed', 'blocks', 'norm']}
        bp, op = [], []
        for name, param in self.named_parameters():
            if '.'.join(name.split('.')[:3]) in bb: bp.append(param)
            else: op.append(param)
        return [{'params':bp,'lr':lr*backbone_scale},{'params':op,'lr':lr}]

# ============================================================

## Training

In [ ]:
def set_backbone(model, req):
    for m in [model.vit.patch_embed, model.vit.blocks, model.vit.norm]:
        for p in m.parameters(): p.requires_grad = req

def phase1_opt(model):
    set_backbone(model, False)
    params = [model.vit.learnable_token, model.vit.pos_embed]
    params += list(model.head.parameters())
    return torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

def phase2_opt(model):
    set_backbone(model, True)
    return torch.optim.AdamW(
        model.param_groups(LR, BACKBONE_SCALE), weight_decay=WEIGHT_DECAY)

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval(); total = 0.
    for img, csi_gt in loader:
        img, csi_gt = img.to(DEVICE), csi_gt.to(DEVICE)
        with torch.amp.autocast('cuda'):
            total += criterion(model(img), csi_gt).item()
    return total / len(loader)


model     = SingleViTCSI(PRETRAINED_PATH).to(DEVICE)
scaler    = GradScaler()
criterion = nn.MSELoss()
best_val  = float('inf')

opt = phase1_opt(model)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE1_EPOCHS, eta_min=LR*0.01)

for epoch in range(1, TOTAL_EPOCHS + 1):
    if epoch == PHASE1_EPOCHS + 1:
        opt = phase2_opt(model)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=PHASE2_EPOCHS, eta_min=LR*BACKBONE_SCALE*0.01)

    model.train(); tloss = 0.
    for img, csi_gt in train_loader:
        img, csi_gt = img.to(DEVICE), csi_gt.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = criterion(model(img), csi_gt)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tloss += loss.item()

    tloss /= len(train_loader)
    vloss  = validate(model, val_loader, criterion)
    sch.step()

    if vloss < best_val:
        best_val = vloss
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'val_loss': vloss}, os.path.join(SAVE_DIR, CKPT_NAME))